# 03 - 生成器与 Provider 体系

> **何时使用**: 当你需要选择数据生成引擎（base/faker/mimesis），或者了解 31 种生成器的能力时。
>
> **核心概念**: sqlseed 支持 3 种 Provider，按丰富度降级：mimesis（默认推荐）→ faker → base（零依赖）。

## 适用场景

- CI/CD 环境不想安装额外依赖 → 使用 `base`
- 需要丰富的本地化数据（中文姓名、地址等）→ 使用 `mimesis`
- 需要特定格式的数据（如 SSN、车牌号）→ 查看 31 种生成器
- 想开发自定义生成器 → 实现 `DataProvider` Protocol

## 你将学到

- 31 种内置生成器类型
- 3 种 Provider 的能力差异和降级策略
- locale 本地化支持
- 自定义 Provider 开发

**📚 教程导航**

| 序号 | 主题 | 架构层 | 前置要求 |
|------|------|--------|----------|
| 01 | 快速上手与核心流程 | Orchestrator | 无 |
| 02 | 9 级策略链详解 | Core: ColumnMapper | 01 |
| **→ 03** | **生成器与 Provider 体系** | **Generators** | **01** |
| 04 | 数据库层与多表关联 | Database + Core | 01 |
| 05 | 表达式派生与约束求解 | Core: DAG / Expression | 01 |
| 06 | 配置驱动与 Transform | Config / Core | 01 |
| 07 | AI 智能配置 | Plugins: AI | 01 |
| 08 | MCP 服务器集成 | Plugins: MCP | 07 |
| 09 | 插件系统与 Hook 生命周期 | Plugins | 01 |
| 10 | CLI 参考手册 | CLI | 06 |
| 11 | 工具类参考 | Utils | 01 |
| 12 | 测试集成模式 | Testing | 01 |

---

In [1]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys; sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 架构定位

| 模块 | 文件 | 核心类/函数 |
|------|------|------------|
| 生成器中心 | `src/sqlseed/generators/registry.py` | `GeneratorRegistry` |

> 对应架构图: [§4 数据生成层架构](../docs/architecture.zh-CN.md#4-数据生成层架构)

## 1. 先看效果 — 三种 Provider 对比

同一个表、同一个列，不同 Provider 生成的数据质量差异一目了然：

In [2]:
for provider_name in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=provider_name)
    sep = "=" * 70
    print()
    print(sep)
    print(f"  Provider: {provider_name}")
    print(sep)
    for row in rows:
        addr = str(row.get("address", "N/A"))[:40]
        print(f"  name={row.get('name', 'N/A'):<20s} email={row.get('email', 'N/A'):<30s}")
        print(f"  phone={row.get('phone', 'N/A'):<20s} address={addr}")


  Provider: base
  name=Sharon Martin        email=cynthia.white364@sample.dev   
  phone=843-338-9697         address=8483 Maple Dr, Springfield, PA
  name=Ryan Adams           email=margaret.baker355@example.com 
  phone=308-867-1045         address=3719 Maple Dr, Portland, PA

  Provider: faker
  name=Micheal Stewart      email=carrieramsey@example.org      
  phone=367.580.7384x0668    address=27294 Ronald Cape, Reeseland, CA 15745
  name=Misty Stanton        email=heather76@example.com         
  phone=443.851.4460         address=59695 White Courts Suite 343, Christinev

  Provider: mimesis
  name=Fidel Camacho        email=dig1888@example.org           
  phone=+17698016227         address=504 Coldspring Walk
  name=Josef Estrada        email=clusters1817@yahoo.com        
  phone=+13649258871         address=561 Soule Extension


- **base**: 无外部依赖，生成随机字符串 — 适合 CI 环境
- **faker**: 社区生态丰富，方法多 — 适合开发调试
- **mimesis**: 本地化最好，性能最高 — **默认推荐**

下面详细拆解每种生成器。

## 2. 31 种生成器全览

sqlseed 内置 31 种生成器，分为三大类：

### 基础类型（7 种）

| 生成器 | 说明 | 关键参数 |
|--------|------|----------|
| `string` | 随机字符串 | min_length, max_length, charset |
| `integer` | 整数 | min_value, max_value |
| `float` | 浮点数 | min_value, max_value, precision |
| `boolean` | 布尔值 | - |
| `bytes` | 二进制数据 | length |
| `json` | JSON 数据 | - |
| `choice` | 枚举选择 | choices |

### 语义类型（17 种）

| 生成器 | 说明 | 生成示例 |
|--------|------|----------|
| `name` | 姓名 | 张三 / John Smith |
| `first_name` | 名 | 伟 / John |
| `last_name` | 姓 | 王 / Smith |
| `username` | 用户名 | user_3847 |
| `email` | 邮箱 | test@example.com |
| `phone` | 电话 | +1-555-0123 |
| `address` | 地址 | 123 Main St |
| `city` | 城市 | Beijing / New York |
| `country` | 国家 | China / United States |
| `state` | 省/州 | California |
| `zip_code` | 邮编 | 10001 |
| `company` | 公司 | Acme Inc |
| `job_title` | 职位 | Software Engineer |
| `url` | 网址 | https://example.com |
| `ipv4` | IP 地址 | 192.168.1.1 |
| `uuid` | UUID | 550e8400-e29b-41d4... |
| `country_code` | 国家代码 | CN / US |

### 时间/文本类型（7 种）

| 生成器 | 说明 | 生成示例 |
|--------|------|----------|
| `date` | 日期 | 2024-01-15 |
| `datetime` | 日期时间 | 2024-01-15 10:30:00 |
| `timestamp` | 时间戳 | 1705312200 |
| `text` | 长文本 | Lorem ipsum... |
| `sentence` | 短句 | The quick brown fox... |
| `password` | 密码 | k8Xf2mPq |
| `pattern` | 正则生成 | PRJ-000123 |

| 特性 | BaseProvider | FakerProvider | MimesisProvider |
|------|:-----------:|:------------:|:--------------:|
| 依赖 | 无 | faker | mimesis |
| 本地化 | ❌ | ✅ (en_US, zh_CN...) | ✅ (en, zh...) |
| 语义质量 | 基础随机 | 高 | 最高 |
| 速度 | 最快 | 中 | 快 |
| 安装 | 默认 | `pip install sqlseed[faker]` | `pip install sqlseed[mimesis]` |

## 3. 6 个代表性生成器深度演示

### 3.1 email — 语义推断

`email` 生成器会根据 locale 生成不同风格的邮箱地址。

In [3]:
from sqlseed import preview

rows = preview(str(db_path), table="members", count=3, provider="mimesis")
for row in rows:
    print(f"email: {row['email']}")

email: hepatitis1900@duck.com
email: ppc1955@duck.com
email: switches2071@example.com


### 3.2 pattern — 正则生成

`pattern` 生成器使用 `rstr` 库按正则表达式生成数据，适合有固定格式的编号。

In [4]:
rows = preview(
    str(db_path),
    table="projects",
    count=5,
    columns={
        "project_no": {"type": "pattern", "regex": "PRJ-\\d{6}"},
    },
)
for row in rows:
    print(f"project_no: {row['project_no']}")

project_no: PRJ-703549
project_no: PRJ-948784
project_no: PRJ-852110
project_no: PRJ-090002
project_no: PRJ-385638


### 3.3 choice — 枚举选择

`choice` 生成器从给定选项中随机选择，适合状态、类型等有限集合。

In [5]:
rows = preview(
    str(db_path),
    table="tasks",
    count=5,
    columns={
        "priority": {"type": "choice", "choices": [1, 2, 3, 4]},
        "status": {"type": "choice", "choices": [0, 1, 2, 3]},
    },
)
for row in rows:
    print(f"priority: {row['priority']}, status: {row['status']}")

priority: 2, status: 3
priority: 1, status: 3
priority: 3, status: 2
priority: 3, status: 3
priority: 2, status: 2


### 3.4 null_ratio — 空值控制

`null_ratio` 参数控制空值的生成比例（0.0-1.0）。

In [6]:
import sqlite3

from sqlseed import ColumnConfig

# null_ratio 需要通过 ColumnConfig 对象传递,使用 connect() API
with connect(str(db_path)) as orch:
    result = orch.fill_table("members", count=20, clear_before=True,
        column_configs=[
            ColumnConfig(name="phone", generator="phone", null_ratio=0.3),
            ColumnConfig(name="address", generator="address", null_ratio=0.5),
        ])


conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT phone, address FROM members").fetchall()
phone_null = sum(1 for r in rows if r[0] is None)
addr_null = sum(1 for r in rows if r[1] is None)
print(f"phone null: {phone_null}/20 ({phone_null/20*100:.0f}%)")
print(f"address null: {addr_null}/20 ({addr_null/20*100:.0f}%)")
conn.close()

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

phone null: 5/20 (25%)
address null: 15/20 (75%)


### 3.5 faker.name — Provider 切换

通过 `provider` 参数在列级别切换 Provider。`native_faker_method` 可以调用 Faker 的原生方法。

In [7]:
rows = preview(
    str(db_path),
    table="members",
    count=3,
    provider="faker",
    locale="zh_CN",
)
for row in rows:
    print(f"name: {row['name']}, email: {row['email']}")
print("\nFaker + zh_CN locale 生成中文姓名和邮箱")

name: 陈淑兰, email: wei22@example.net
name: 潘斌, email: pingmo@example.org
name: 谭秀云, email: rsong@example.net

Faker + zh_CN locale 生成中文姓名和邮箱


### 3.6 mimesis.address — Mimesis 对比

Mimesis 是默认 Provider，本地化支持最好。注意 locale 格式差异：
- Faker: `en_US`, `zh_CN`（带下划线）
- Mimesis: `en`, `zh`（短代码）

In [8]:
rows = preview(
    str(db_path),
    table="members",
    count=3,
    provider="mimesis",
    locale="zh",
)
for row in rows:
    print(f"name: {row['name']}, address: {row.get('address', 'N/A')}")
print("\nMimesis + zh locale 生成中文姓名和地址")

name: 恒世 上官, address: 北湖三条1302号
name: 晨濡 宗政, address: 柿铺六条325号
name: 虹 汲, address: 七里河侧路393号

Mimesis + zh locale 生成中文姓名和地址


## 4. Provider 降级策略

当首选 Provider 不可用时，sqlseed 自动降级：

```
mimesis → faker → base
```

- 如果 `mimesis` 未安装，自动降级到 `faker`
- 如果 `faker` 也未安装，降级到 `base`
- `base` 始终可用，零依赖

降级是静默的，不会报错，但生成质量会降低。

In [9]:
for provider_name in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=provider_name)
    print(f"\n--- {provider_name} ---")
    for row in rows:
        print(f"  name={row['name']}, email={row['email']}")


--- base ---
  name=Laura Adams, email=linda.gomez552@mail.net
  name=Joseph Carter, email=george.robinson356@mail.net

--- faker ---
  name=Joshua Zimmerman, email=allen08@example.com
  name=Scott Austin, email=mguzman@example.com

--- mimesis ---
  name=Sylvie Dillon, email=slow1813@outlook.com
  name=Elayne Evans, email=committees2087@duck.com


## 5. locale 与 seed

### locale 设置

| Provider | 支持的 locale 格式 | 示例 |
|----------|-------------------|------|
| mimesis | 短代码 | `en`, `zh`, `ja`, `de` |
| faker | 下划线代码 | `en_US`, `zh_CN`, `ja_JP` |
| base | 无本地化 | - |

### seed 可复现

In [10]:
rows1 = preview(str(db_path), table="members", count=3, seed=42)
rows2 = preview(str(db_path), table="members", count=3, seed=42)

names1 = [r['name'] for r in rows1]
names2 = [r['name'] for r in rows2]

print(f"Run 1: {names1}")
print(f"Run 2: {names2}")
print(f"Reproducible: {names1 == names2}")

Run 1: ['Anthony Reilly', 'Jaye Hunt', 'Randy Lynch']
Run 2: ['Anthony Reilly', 'Jaye Hunt', 'Randy Lynch']
Reproducible: True


## 🆕 bytes / json / timestamp 生成器

这三种生成器用于特殊数据类型：

In [11]:
with sqlseed.connect(str(db_path)) as orch:
    preview_bytes = orch.preview_table("organizations", count=3, columns={
        "description": {"generator": "bytes"},
    })
    print("bytes 生成器 (raw bytes):")
    for row in preview_bytes:
        val = row.get('description')
        print(f"  description type: {type(val).__name__}, len: {len(val) if val else 0}")

preview_json = sqlseed.preview(str(db_path), table="organizations", count=2, columns={
    "description": {"generator": "json"}
})
print("\njson 生成器:")
for row in preview_json:
    print(f"  {row.get('description', 'N/A')}")

preview_ts = sqlseed.preview(str(db_path), table="organizations", count=2, columns={
    "created_at": {"generator": "timestamp"}
})
print("\ntimestamp 生成器:")
for row in preview_ts:
    print(f"  created_at: {row.get('created_at')}")

bytes 生成器 (raw bytes):
  description type: bytes, len: 16
  description type: bytes, len: 16
  description type: bytes, len: 16

json 生成器:
  {"id": 839617, "name": "Orval Hunter", "active": true}
  {"id": 181944, "name": "Maximo King", "active": false}

timestamp 生成器:
  created_at: 1786317696
  created_at: 1778608156


## 🔧 底层 Provider 原生方法

sqlseed 的 Provider 封装了 Faker 和 Mimesis 库。高级用户可以通过 ProviderRegistry 访问底层库的原生方法，获取内置生成器未覆盖的数据类型（如 `license_plate`、`credit_card_number`、`food.fruit` 等）。

In [12]:
from sqlseed.generators.registry import ProviderRegistry

# 底层 Provider 可以直接调用原生方法
# 这是高级用法,一般通过 columns={} 配置即可
registry = ProviderRegistry()

# Faker 原生方法
registry.ensure_provider("faker")
faker_provider = registry.get("faker")
faker_provider.set_locale("en_US")
faker_obj = getattr(faker_provider, "_faker", None)

print("Faker 原生方法示例:")
print(f"  company_suffix: {faker_obj.company_suffix()}")
print(f"  catch_phrase:   {faker_obj.catch_phrase()}")
print(f"  bs:             {faker_obj.bs()}")
print(f"  license_plate:  {faker_obj.license_plate()}")

# Mimesis 原生方法
registry.ensure_provider("mimesis")
mimesis_provider = registry.get("mimesis")
generic_obj = getattr(mimesis_provider, "_generic", None)

print("\nMimesis 原生方法示例:")
print(f"  text.word:      {generic_obj.text.word()}")
print(f"  person.title:   {generic_obj.person.title()}")
print(f"  food.fruit:     {generic_obj.food.fruit()}")
print(f"  science.metric: {generic_obj.science.metric_prefix()}")

Faker 原生方法示例:
  company_suffix: and Sons
  catch_phrase:   Face-to-face national analyzer
  bs:             cultivate intuitive e-business
  license_plate:  934 CAV

Mimesis 原生方法示例:
  text.word:      technique
  person.title:   LL.D
  food.fruit:     Honeydew
  science.metric: giga


## 📦 ProviderRegistry 完整 API

ProviderRegistry 管理所有数据生成 Provider，支持注册、查询、设置默认等操作。

In [13]:
from sqlseed.generators.registry import ProviderRegistry

registry = ProviderRegistry()
print(f"默认 Provider: {registry.default_name}")
print(f"可用 Providers: {registry.available_providers}")

base = registry.get("base")
print(f"\nBaseProvider: name={base.name}")

registry.ensure_provider("faker")
print(f"加载 FakerProvider 后: {registry.available_providers}")

registry.ensure_provider("mimesis")
print(f"加载 MimesisProvider 后: {registry.available_providers}")

registry.set_default("faker")
print(f"切换默认为 faker: default_name={registry.default_name}")

默认 Provider: base
可用 Providers: ['base']

BaseProvider: name=base
加载 FakerProvider 后: ['base', 'faker']
加载 MimesisProvider 后: ['base', 'faker', 'mimesis']
切换默认为 faker: default_name=faker


## 🔄 Provider 切换

通过 `preview()` / `fill()` 的 `provider` 参数切换生成引擎。不同 Provider 的数据质量和本地化支持差异明显：

In [14]:
# Provider 切换通过 preview/fill 的 provider 参数实现
# 不同 Provider 生成同一类型数据的质量差异：

print("Provider 对比 — 同一列,不同引擎:\n")
print(f"{'Provider':<10s}  {'name':<25s}  {'email':<30s}")
print("-" * 68)

for prov in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=prov)
    for row in rows:
        print(f"{prov:<10s}  {row['name']:<25s}  {row['email']:<30s}")
    print()

print("建议: 开发调试用 faker,CI 用 base,生产数据用 mimesis")

Provider 对比 — 同一列,不同引擎:

Provider    name                       email                         
--------------------------------------------------------------------
base        Amanda Robinson            jeffrey.gomez439@demo.io      
base        Donna Williams             betty.nelson423@demo.io       

faker       Olivia Quinn               susan65@example.net           
faker       Patrick Tyler              nreeves@example.net           

mimesis     Wanetta Pena               duty1810@gmail.com            
mimesis     Weldon Harper              effectiveness1944@example.com 

建议: 开发调试用 faker,CI 用 base,生产数据用 mimesis


## 6. 总结

| 要点 | 说明 |
|------|------|
| 31 种生成器 | 基础 7 + 语义 17 + 时间/文本 7 |
| 3 种 Provider | mimesis（默认）> faker > base |
| 自动降级 | 缺依赖时静默降级，不报错 |
| locale | mimesis 用短代码，faker 用下划线代码 |
| seed | 设置后可复现，适合测试 |
| null_ratio | 0.0-1.0 控制空值比例 |

**下一步**: [04-database-advanced.ipynb](04-database-advanced.ipynb) — 数据库层与多表关联

In [15]:
# ✅ 验证: 确保数据已成功生成并写入
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # 基本行数验证
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
